In [1]:
!sudo apt-get update
!sudo apt-get install -y pciutils

Get:1 http://archive.ubuntu.com/ubuntu jammy InRelease [270 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]        
Get:3 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]      
Get:4 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:5 https://ppa.launchpadcontent.net/git-core/ppa/ubuntu jammy InRelease [24.6 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]      
Get:7 http://archive.ubuntu.com/ubuntu jammy/universe amd64 Packages [17.5 MB] 
Get:8 https://apt.postgresql.org/pub/repos/apt jammy-pgdg InRelease [129 kB]   
Get:9 http://archive.ubuntu.com/ubuntu jammy/multiverse amd64 Packages [266 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy/restricted amd64 Packages [164 kB]
Get:11 http://archive.ubuntu.com/ubuntu jammy/main amd64 Packages [1,792 kB]
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,535 kB]
Get:13 http://archive.

In [2]:
# from https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/latest/install-guide.html#installation
# !curl -fsSL https://nvidia.github.io/libnvidia-container/gpgkey | sudo gpg --dearmor -o /usr/share/keyrings/nvidia-container-toolkit-keyring.gpg \
#   && curl -s -L https://nvidia.github.io/libnvidia-container/stable/deb/nvidia-container-toolkit.list | \
#     sed 's#deb https://#deb [signed-by=/usr/share/keyrings/nvidia-container-toolkit-keyring.gpg] https://#g' | \
#     sudo tee /etc/apt/sources.list.d/nvidia-container-toolkit.list
# !sudo apt-get update
# !sudo apt-get install -y nvidia-container-toolkit

In [3]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%                  4.7%                                     6.7%########################                                58.3%##############################################                80.9%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> NVIDIA GPU installed.
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


Do not forget the `ollama serve` on a terminal, also need to pull a model

In [36]:
# !ollama pull smollm2:1.7b
!ollama pull llama3.2:3b
# !ollama pull qwen2-math:1.5b

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest 
pulling dde5aa3fc5ff... 100% ▕████████████████▏ 2.0 GB                         
pulling 966de95ca8a6... 100% ▕████████████████▏ 1.4 KB                         
pulling fcc5a6bec9da... 100% ▕████████████████▏ 7.7 KB                         
pulling a70ff7e570d9... 100% ▕████████████████▏ 6.0 KB                         
pulling 56bb8bd477a5... 100% ▕████████████████▏   96 B                         
pulling 34bb5ab01051... 100% ▕████████████████▏  561 B                         
verifying sha256 digest 
writing manifest 
success 


In [7]:
!pip install ollama langchain transformers[torch] datasets trl "accelerate>=0.26.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 86.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 92.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 767.5/767.5 kB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.9/796.9 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 112.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 95.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 84.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 613.1/613.1 kB 52.1 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.2.0
    Uninstalling fsspec-2025.2.0:
      Successfully uninstalled fsspec-2025.2.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the 

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from trl import SFTConfig, SFTTrainer, setup_chat_format
import torch
import os

device = (
    "cuda"
    if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available() else "cpu"
)

# Load the model and tokenizer
model_name = "HuggingFaceTB/SmolLM2-360M"
model = AutoModelForCausalLM.from_pretrained(
    pretrained_model_name_or_path=model_name
)
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_name)

# Set up the chat format
model, tokenizer = setup_chat_format(model=model, tokenizer=tokenizer)

/opt/conda/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
from datasets import load_dataset

dataset = load_dataset('gsm8k', 'main')
def tokenize_function(examples):
    examples["text"] = tokenizer.apply_chat_template([{"role": "user", "content": examples["question"].strip()}, {"role": "assistant", "content": examples["answer"].strip()}], tokenize=False)
    return examples
ds = dataset.map(tokenize_function)
print(ds["train"]["text"][0])

Map: 100%|██████████| 1319/1319 [00:00<00:00, 7897.05 examples/s]

<|im_start|>user
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?<|im_end|>
<|im_start|>assistant
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72<|im_end|>



In [6]:
ds["train"]["question"][0]

'Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?'

In [18]:
from transformers import pipeline

pipe_default = pipeline("text-generation", model=model, tokenizer=tokenizer)
print(pipe_default(ds["test"]["question"][0], max_new_tokens=200, return_full_text=True)[0]["generated_text"])

Device set to use cuda:0


Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

Solution:

Let x be the number of ducks that Janet buys.

The number of eggs she buys is 16.

The number of ducks she sells is 3.

The number of ducks she eats is 4.

The number of ducks she bakes is 4.

The number of dollars she makes at the farmers' market is $2 x + 3 x + 4 x + 4 = 2 x + 3 x + 4 x + 4 = 2 x + 3 x + 4 x + 4 = 2 x + 3 x + 4 x + 4 = 2 x + 3 x + 4 x + 4 = 2 x + 3 x + 4 x + 4 = 2 x + 3 x + 4 x + 4 = 2 x + 3 x + 4 x + 4


In [19]:
print(ds["test"]["answer"][0])

Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18


In [ ]:
os.environ["PYTORCH_MPS_HIGH_WATERMARK_RATIO"] = "0.0"

# Configure the SFTTrainer
sft_config = SFTConfig(
    output_dir="../outputs/sft_output",
    num_train_epochs=1,
    per_device_train_batch_size=4,  # Set according to your GPU memory capacity
    learning_rate=5e-5,  # Common starting point for fine-tuning
    logging_steps=100,  # Frequency of logging training metrics
    use_mps_device= True if device == "mps" else False,
    hub_model_id="argilla/SmolLM2-360M-synthetic-concise-reasoning",  # Set a unique name for your model
    push_to_hub=False,
)

# Initialize the SFTTrainer
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=ds["train"],
    tokenizer=tokenizer,
)


/tmp/ipykernel_24339/1219757310.py:16: FutureWarning: `tokenizer` is deprecated and removed starting from version 0.16.0 for `SFTTrainer.__init__`. Use `processing_class` instead.
  trainer = SFTTrainer(


In [6]:
trainer.train()

Step,Training Loss
100,1.082500
200,1.006600
300,0.981800
400,0.978400
500,0.973400
600,0.963800
700,0.936500
800,0.943400
900,0.944100
1000,0.937900


TrainOutput(global_step=1869, training_loss=0.9471341638784475, metrics={'train_runtime': 2012.2113, 'train_samples_per_second': 3.714, 'train_steps_per_second': 0.929, 'total_flos': 3837263211302400.0, 'train_loss': 0.9471341638784475})

In [9]:
checkpoint_path = "sft_output/checkpoint-1869"

# Load model and tokenizer
sft_model = AutoModelForCausalLM.from_pretrained(checkpoint_path)
sft_pipe = pipeline("text-generation", model=sft_model, tokenizer=tokenizer)

Device set to use cuda:0


In [20]:
print(sft_pipe(ds["test"]["question"][0], max_new_tokens=150, early_stopping = True)[0]["generated_text"])

/opt/conda/lib/python3.12/site-packages/transformers/generation/configuration_utils.py:677: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(


Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?

First find the total number of ducks that Janet has: 16 eggs/day * 3 days/egg = <<16*3=48>>48 ducks
Then find the number of ducks that she sells: 48 ducks - 3 ducks = <<48-3=45>>45 ducks
Then find the number of ducks that she eats: 45 ducks * 3 ducks/day = <<45*3=135>>135 ducks
Then find the number of ducks that she bakes: 135 ducks * 4 ducks/day = <<135*4=540>>540 ducks
Then find the number


In [ ]:
ans = sft_pipe("I have 2 apples, Tim gives me 2 more, and then I give 1 to Ana. How much do I have left? Let's think step by step.", return_full_text=False, max_new_tokens=100)
print(ans[0]["generated_text"])

Device set to use cuda:0


I have 2 apples, Tim gives me 2 more, and then I give 1 to Ana. How much do I have left? Let's think step by step. We know that Tim gave me 2 apples, so I have 2 + 2 = <<2+2=4>>4 apples.
Then Tim gave me 2 more, so I have 4 + 2 = <<4+2=6>>6 apples.
And then I gave 1 to Ana, so I have 6 - 1 = <<6-1=5>>5 apples.
#### 5 apples

#### 5 apples

####


In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load the fine-tuned model
model_name = "sft_output/checkpoint-1869"  # Change to your fine-tuned model name
device = "cuda" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
tokenizer = AutoTokenizer.from_pretrained(model_name)

/opt/conda/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
!pip install langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 94.2 MB/s eta 0:00:00


In [ ]:
from langchain.llms import HuggingFacePipeline
from transformers import pipeline


# Create a text generation pipeline
hf_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_length=512,
    do_sample=False,
    # temperature=0.0,
)

# Wrap it in LangChain
llm = HuggingFacePipeline(pipeline=hf_pipeline)

# Test it
print(llm("Solve: What is 12 * 7 + 5?"))

Device set to use cuda:0


Solve: What is 12 * 7 + 5?
12 * 7 + 5 = <<12*7+5=85>>85
#### 85

#### 12 * 7 + 5 = 85

#### 12 * 7 + 5 = 85

#### 12 * 7 + 5 = 85

#### 12 * 7 + 5 = 85

#### 12 * 7 + 5 = 85

#### 12 * 7 + 5 = 85

#### 12 * 7 + 5 = 85

#### 12 * 7 + 5 = 85

#### 12 * 7 + 5 = 85

#### 12 * 7 + 5 = 85

#### 12 * 7 + 5 = 85

#### 12 * 7 + 5 = 85

#### 12 * 7 + 5 = 85

#### 12 * 7 + 5 = 85

#### 12 * 7 + 5 = 85

#### 12 * 7 + 5 = 85

#### 12 * 7 + 5 = 85

#### 12 * 7 + 5 = 85

#### 12 * 7 + 5 = 85

#### 12 * 7 + 5 = 85

#### 12 * 7 + 5 = 85

#### 12 * 7 + 5 = 85

#### 12 * 7 + 5 = 85

#### 12 * 7 + 5 = 85

#### 12 * 7 + 5 = 85

#### 12 * 7 + 5 = 85

#### 12 * 7 + 5 = 85

#### 12 * 7 + 5 = 85

#### 12 * 7 + 5 = 85

#### 1


In [32]:
from langchain.tools import Tool

def calculator_tool(expression):
    try:
        return eval(expression, {"__builtins__": {}})
    except Exception as e:
        return str(e)

calculator = Tool(
    name="calculator",
    func=calculator_tool,
    description="Evaluates a mathematical expression."
)

In [ ]:
from langchain.llms import Ollama

# Load the model from Ollama
llm = Ollama(model="llama3.2:3b")  # Change to your fine-tuned model name

# Test it
question = "Mark's basketball team scores 25 2 pointers, 8 3 pointers and 10 free throws.  Their opponents score double the 2 pointers but half the 3 pointers and free throws.  What's the total number of points scored by both teams added together?"
# question = "Angelo and Melanie want to plan how many hours over the next week they should study together for their test next week. They have 2 chapters of their textbook to study and 4 worksheets to memorize. They figure out that they should dedicate 3 hours to each chapter of their textbook and 1.5 hours for each worksheet. If they plan to study no more than 4 hours each day, how many days should they plan to study total over the next week if they take a 10-minute break every hour, include 3 10-minute snack breaks each day, and 30 minutes for lunch each day?"

# question = "I have 3 apples, Ana gives me 2 more, I eat 1 and then I give all the rest to Bob. How much apples does Bob have?"
print(llm("hi"))

KeyboardInterrupt: 

In [31]:
from langchain.agents import initialize_agent, AgentType
from langchain.memory import ConversationBufferMemory
from langchain.prompts import PromptTemplate

memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)


# custom_prompt = PromptTemplate.from_template(
#     """
#     You are an AI assistant that follows the ReAct framework:  
#     - **Thought**: Think step-by-step before taking an action.  
#     - **Action**: Call a tool if necessary.  
#     - **Observation**: Wait for the tool result.  
#     - **Final Answer**: Provide the result.  
   
#     User Question: {input}  
   
#     {agent_scratchpad}  
#     """
# )

agent = initialize_agent(
    tools=[calculator],  # The tool your model can use
    llm=llm,  # Your fine-tuned Hugging Face model
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True,
    handle_parsing_errors=True,
    memory=memory, 
    # agent_kwargs={"prompt": custom_prompt}  # Force correct reasoning format
)

# Test the model with external calculation

#"I have 3 apples, Ana gives me 2 more, I eat 1 and then I give all the rest to Bob. How much apples does Bob have?"
# question = "Angelo and Melanie want to plan how many hours over the next week they should study together for their test next week. They have 2 chapters of their textbook to study and 4 worksheets to memorize. They figure out that they should dedicate 3 hours to each chapter of their textbook and 1.5 hours for each worksheet. If they plan to study no more than 4 hours each day, how many days should they plan to study total over the next week if they take a 10-minute break every hour, include 3 10-minute snack breaks each day, and 30 minutes for lunch each day?"
print(agent.run(question))



> Entering new AgentExecutor chain...
Question: Mark's basketball team scores 25 2 pointers, 8 3 pointers and 10 free throws. Their opponents score double the 2 pointers but half the 3 pointers and free throws. What's the total number of points scored by both teams added together?

Thought: First, I need to calculate the total points scored by Mark's team.

Action: calculator
Action Input: (25 * 2) + (8 * 3) + (10 * 1)
Observation: 84
Thought:Thought: Now that I know the total points scored by Mark's team, I need to calculate the total points scored by their opponents.

Action: calculator
Action Input: 2 * (25 * 2) + 0.5 * (8 * 3) + 0.5 * (10 * 1)
Observation: 117.0
Thought:Thought: The total points scored by Mark's team and their opponents are now known.

Action: calculator
Action Input: 84 + 117.0
Observation: 201.0
Thought:Final Answer: The total number of points scored by both teams added together is 201.0.

> Finished chain.
The total number of points scored by both teams added 